# 01 · Carga y entendimiento de datos
Tabla: `clientes` (capa bronze)

## 1. Leer los datos

In [1]:
import pandas as pd
from pathlib import Path

ruta = Path("../../data/bronze_simple/data/bronze_simple/clientes/clientes.csv")

df = pd.read_csv(ruta, parse_dates=["fecha_vinculacion"])

print(df.shape)
df.head()

(53, 6)


,cliente_id,fecha_vinculacion,tipo_cliente,rango_ingreso,departamento,antiguedad_laboral_meses
0,C00001,2023-02-04,INDEPENDIENTE,8M+,SANTANDER,-68
1,C00002,2017-07-25,DEPENDIENTE,1-2M,CUNDINAMARCA,164
2,C00003,2018-07-13,DEPENDIENTE,8M+,SANTANDER,91
3,C00004,2020-04-09,DEPENDIENTE,1-2M,CUNDINAMARCA,127
4,C00005,2020-04-26,DEPENDIENTE,5-8M,VALLE,84


## 2. Estructura y perfil

In [3]:
df.info()
print("Nulos por columna:")
print(df.isna().sum())
print("Filas duplicadas:", df.duplicated().sum())
df.describe(include="all").T

<class 'pandas.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   cliente_id                53 non-null     str           
 1   fecha_vinculacion         53 non-null     datetime64[us]
 2   tipo_cliente              53 non-null     str           
 3   rango_ingreso             49 non-null     str           
 4   departamento              53 non-null     str           
 5   antiguedad_laboral_meses  53 non-null     int64         
dtypes: datetime64[us](1), int64(1), str(4)
memory usage: 2.6 KB
Nulos por columna:
cliente_id                  0
fecha_vinculacion           0
tipo_cliente                0
rango_ingreso               4
departamento                0
antiguedad_laboral_meses    0
dtype: int64
Filas duplicadas: 3


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
cliente_id,53,50,C00006,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fecha_vinculacion,53,NaN,NaN,NaN,2019-04-26 14:56:36.226415,2015-12-08 00:00:00,2017-07-03 00:00:00,2018-09-18 00:00:00,2020-10-27 00:00:00,2023-04-20 00:00:00,NaN
tipo_cliente,53,2,DEPENDIENTE,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rango_ingreso,49,5,5-8M,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
departamento,53,5,ATLANTICO,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
antiguedad_laboral_meses,53.0,NaN,NaN,NaN,79.962264,-163.0,49.0,84.0,130.0,175.0,61.374626


## 3. Problemas de calidad

In [2]:
# IDs repetidos
df[df["cliente_id"].duplicated(keep=False)].sort_values("cliente_id")

,cliente_id,fecha_vinculacion,tipo_cliente,rango_ingreso,departamento,antiguedad_laboral_meses
5,C00006,2016-11-18,DEPENDIENTE,5-8M,ATLANTICO,50
51,C00006,2016-11-18,DEPENDIENTE,5-8M,ATLANTICO,50
42,C00043,2018-02-27,INDEPENDIENTE,2-3M,ATLANTICO,19
50,C00043,2018-02-27,INDEPENDIENTE,2-3M,ATLANTICO,19
47,C00048,2016-08-08,INDEPENDIENTE,2-3M,ANTIOQUIA,77
52,C00048,2016-08-08,INDEPENDIENTE,2-3M,ANTIOQUIA,77


In [3]:
# Antigüedad laboral negativa
df[df["antiguedad_laboral_meses"] < 0]

,cliente_id,fecha_vinculacion,tipo_cliente,rango_ingreso,departamento,antiguedad_laboral_meses
0,C00001,2023-02-04,INDEPENDIENTE,8M+,SANTANDER,-68
19,C00020,2020-03-06,DEPENDIENTE,1-2M,ANTIOQUIA,-163


In [4]:
# Nulos en rango_ingreso
df[df["rango_ingreso"].isna()]

,cliente_id,fecha_vinculacion,tipo_cliente,rango_ingreso,departamento,antiguedad_laboral_meses
9,C00010,2023-01-21,INDEPENDIENTE,NaN,ATLANTICO,141
15,C00016,2017-06-20,DEPENDIENTE,NaN,SANTANDER,51
20,C00021,2019-10-10,INDEPENDIENTE,NaN,ANTIOQUIA,25
32,C00033,2023-01-27,DEPENDIENTE,NaN,VALLE,6


## 4. Categorías y fechas

In [6]:
for col in ["tipo_cliente", "rango_ingreso", "departamento"]:
    print(f"{col}")
    print(df[col].value_counts(dropna=False))

print("Rango de fechas:", df["fecha_vinculacion"].min().date(), "->", df["fecha_vinculacion"].max().date())

tipo_cliente
tipo_cliente
DEPENDIENTE      28
INDEPENDIENTE    25
Name: count, dtype: int64
rango_ingreso
rango_ingreso
5-8M    12
3-5M    12
1-2M     9
2-3M     9
8M+      7
NaN      4
Name: count, dtype: int64
departamento
departamento
ATLANTICO       13
VALLE           11
ANTIOQUIA       11
CUNDINAMARCA    10
SANTANDER        8
Name: count, dtype: int64
Rango de fechas: 2015-12-08 -> 2023-04-20
